# Laboratorio 08. Optimizadores: SGD, momentum y Adam

**Pregunta guía:** ¿Cómo modifican los optimizadores la trayectoria de aprendizaje?

## Objetivos
Al finalizar podrás comparar SGD, momentum y Adam con el mismo modelo, datos, inicialización y número de pasos; interpretar una comparación controlada.

## Prerrequisitos y conexión
Aplica el descenso por gradiente del laboratorio 07 a parámetros de un modelo. Se requiere Python básico; cada notebook puede ejecutarse de forma independiente desde un kernel limpio.

## Teoría breve
SGD usa el gradiente actual, $\theta_{t+1}=\theta_t-\eta g_t$. Momentum suaviza la dirección, $v_t=\beta v_{t-1}+g_t$. Adam escala el primer y segundo momento estimados, $\theta_{t+1}=\theta_t-\eta\hat m_t/(\sqrt{\hat v_t}+\epsilon)$. La comparación solo es controlada si se conservan datos e inicialización.

## Problema y datos
Ajustamos la misma regresión pequeña con tres optimizadores y un protocolo idéntico. Todos los datos se generan en el notebook: no se requieren descargas ni archivos locales.

## Experimento conceptual
¿Cuál converge más rápido en este experimento y qué no permite concluir esa comparación? Fija la semilla, ejecuta las celdas en orden y cambia una decisión cada vez.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Preparar datos e inicialización común
Fijamos lote, modelo y parámetros iniciales idénticos para cada alternativa.


In [ ]:
x = torch.linspace(-2, 2, 96, device=device).unsqueeze(1)
y = 0.8 * x**3 - 0.5 * x + 0.15 * torch.randn_like(x)
initial = nn.Linear(1, 1).to(device)
initial_state = {key: value.detach().clone() for key, value in initial.state_dict().items()}
optimizers = {'SGD': lambda params: torch.optim.SGD(params, lr=0.08),
              'Momentum': lambda params: torch.optim.SGD(params, lr=0.08, momentum=0.9),
              'Adam': lambda params: torch.optim.Adam(params, lr=0.03)}
loss_histories = {}


**Resultado e interpretación.** La tarea es deliberadamente la misma para las tres alternativas. No reajustamos hiperparámetros por optimizador; el resultado depende de esta configuración.


### Entrenar y comparar
Aplicamos el mismo ciclo y comparamos MSE final y curvas.


In [ ]:
for name, make_optimizer in optimizers.items():
    model = nn.Linear(1, 1).to(device); model.load_state_dict(initial_state)
    optimizer = make_optimizer(model.parameters()); history = []
    for epoch in range(120):
        prediction = model(x); loss = F.mse_loss(prediction, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); history.append(loss.item())
    loss_histories[name] = history
    print(name, 'MSE final:', round(history[-1], 4))
for name, history in loss_histories.items():
    plt.plot(history, label=name)
plt.yscale('log'); plt.xlabel('Época'); plt.ylabel('MSE train (escala log)'); plt.legend(); plt.show()


**Resultado e interpretación.** Las curvas describen convergencia en esta tarea y con estas tasas. Un ganador aquí no establece superioridad general ni calidad de generalización.


## Limitaciones y conclusiones clave
Los resultados dependen de esta semilla, tamaño de muestra, arquitectura y protocolo. No extrapoles métricas sintéticas a datos reales; separa decisiones de modelado de conclusiones generales. La celda de salida aporta la evidencia numérica y visual para responder la pregunta guía.

- Los parámetros y activaciones determinan cómo se representa la señal.
- La pérdida, el optimizador y el protocolo de evaluación forman parte del experimento.
- Una métrica aislada no describe todos los errores ni garantiza generalización.

## Ejercicios progresivos
1. **Guiado:** Guiado: repite el experimento con una tasa común menor y contrasta las curvas.
2. **Independiente:** Independiente: cambia únicamente la tasa de Adam y registra MSE final.
3. **Desafío:** Desafío: diseña una búsqueda justa que ajuste hiperparámetros usando validación y reserve test.

## Conexión con el siguiente laboratorio
Usa la representación, operación o criterio de evaluación de este laboratorio como punto de partida del siguiente paso de la secuencia.
